## Run sod shock tube test for different mach numbers

In [33]:
import os
import sys
import shutil
import numpy as np

output_dir = "/home/watanabe/Projects/outputs"
sod_dir = "/home/watanabe/Projects/outputs/Sod"

In [34]:
# list of configurations 
mach_arr = np.array([1.5, 2.0, 5.0, 10.0, 25.0, 50.0, 100, 250, 500, 1000])
pleft_arr = np.array([0.81445, 1.90830, 15.35767, 63.498622, 400.51500, 1604.1492, 6418.6865, 40120.448, 160483.88, 641937.62])
tmax_arr = np.array([16, 12, 4.5, 2.5, 1.0, 0.5, 0.3, 0.1, 0.05, 0.03])

For each configuration, copy flash.par and modify pleft, tend, and spacing for saving plot files

In [35]:
for i, mach in enumerate(mach_arr):
    # if i != 0:
    #     continue
    # make directory
    sod_mach_dir = os.path.join(output_dir, f"Sod_mach{mach:.1f}")
    if not os.path.exists(sod_mach_dir):
        os.mkdir(sod_mach_dir)

    # copy flash4 & flash.par to this directory
    shutil.copy(os.path.join(sod_dir, "flash4"), sod_mach_dir)
    shutil.copy(os.path.join(sod_dir, "flash.par"), sod_mach_dir)

    # open flash.par
    flashpar_file = os.path.join(sod_mach_dir, "flash.par")
    with open(flashpar_file, "r") as f:
        par_lines = f.readlines()

    # modify pleft
    pleft_strlist = par_lines[8].split(" ")
    pleft_strlist[-1] = f"{pleft_arr[i]}\n"
    par_lines[8] = " ".join(pleft_strlist)
    # print(par_lines[8])

    # modify xmax, ymax, xmid
    xmid_strlist = par_lines[26].split(" ")
    xmid_strlist[-1] = f"50.0\n"
    par_lines[26] = " ".join(xmid_strlist)

    for ii in [41, 43]:
        coordmax_strlist = par_lines[ii].split(" ")
        coordmax_strlist[-1] = f"100.0\n"
        par_lines[ii] = " ".join(coordmax_strlist)

    # modify tmax
    tmax_strlist = par_lines[74].split(" ")
    tmax_strlist[-1] = f"{tmax_arr[i]}\n"
    par_lines[74] = " ".join(tmax_strlist)

    # modify time interval for checkpoint & plot file
    tstep_chk = tmax_arr[i] / 2
    tcheck_strlist = par_lines[64].split(" ")
    tcheck_strlist[-1] = f"{tstep_chk:.7f}\n"
    par_lines[64] = " ".join(tcheck_strlist)

    tstep_plt = tmax_arr[i] / 10
    tplt_strlist = par_lines[69].split(" ")
    tplt_strlist[-1] = f"{tstep_plt:.7f}\n"
    par_lines[69] = " ".join(tplt_strlist)

    # finally write to flash.par
    with open(flashpar_file, "w") as f:
        f.writelines(par_lines)

In [36]:
bash_tmpl = """#!/bin/bash -l

"""

for i, mach in enumerate(mach_arr):
    bash_tmpl += f"echo mach{mach:.1f}\n"
    bash_tmpl += f"cd /home/watanabe/Projects/outputs/Sod_mach{mach:.1f}\n"
    bash_tmpl += f"rm -rf *sod* *.dump *.log\n"
    bash_tmpl += f"ls /home/watanabe/Projects/outputs/Sod_mach{mach:.1f}\n"
    bash_tmpl += f"mpirun -np 6 ./flash4 > /home/watanabe/Projects/outputs/Sod_mach{mach:.1f}/stdout.log\n"
    bash_tmpl += f"ls /home/watanabe/Projects/outputs/Sod_mach{mach:.1f}\n"

print(bash_tmpl)
with open("run_sod_mach.sh", "w") as f:
    f.write(bash_tmpl)

#!/bin/bash -l

echo mach1.5
cd /home/watanabe/Projects/outputs/Sod_mach1.5
rm -rf *sod* *.dump *.log
ls /home/watanabe/Projects/outputs/Sod_mach1.5
mpirun -np 6 ./flash4 > /home/watanabe/Projects/outputs/Sod_mach1.5/stdout.log
ls /home/watanabe/Projects/outputs/Sod_mach1.5
echo mach2.0
cd /home/watanabe/Projects/outputs/Sod_mach2.0
rm -rf *sod* *.dump *.log
ls /home/watanabe/Projects/outputs/Sod_mach2.0
mpirun -np 6 ./flash4 > /home/watanabe/Projects/outputs/Sod_mach2.0/stdout.log
ls /home/watanabe/Projects/outputs/Sod_mach2.0
echo mach5.0
cd /home/watanabe/Projects/outputs/Sod_mach5.0
rm -rf *sod* *.dump *.log
ls /home/watanabe/Projects/outputs/Sod_mach5.0
mpirun -np 6 ./flash4 > /home/watanabe/Projects/outputs/Sod_mach5.0/stdout.log
ls /home/watanabe/Projects/outputs/Sod_mach5.0
echo mach10.0
cd /home/watanabe/Projects/outputs/Sod_mach10.0
rm -rf *sod* *.dump *.log
ls /home/watanabe/Projects/outputs/Sod_mach10.0
mpirun -np 6 ./flash4 > /home/watanabe/Projects/outputs/Sod_mach10.0/st